# Local SQL Adapter Serving And KV Cache Walkthrough

This notebook shows the local serving path for the current best SQL adapter from start to finish.

It follows the actual repo code path:

- manifest: `experiments/sql/qwen35_0_8b__exp048_storefront_v3_lora_r16_a32_d010.json`
- base model: `Qwen/Qwen3.5-0.8B-Base`
- adapter: `artifacts/sql/qwen35_0_8b__exp048_storefront_v3_lora_r16_a32_d010/adapter`
- endpoint: local vLLM OpenAI-compatible `/v1/completions`
- example case: `storefront_sales_lab_eval_001`

Long-running cells are guarded. Flip `START_VLLM_SERVER` or `REQUIRE_ENDPOINT` only when you want the notebook to start or require the live local endpoint.

In [1]:
from __future__ import annotations

import json
import os
import shlex
import sqlite3
import subprocess
import sys
import time
import urllib.error
import urllib.request
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src/sqlbench_lab").exists():
            return candidate
    raise RuntimeError("Could not find SQLBench Lab repo root")


ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

MANIFEST_PATH = ROOT / "experiments/sql/qwen35_0_8b__exp048_storefront_v3_lora_r16_a32_d010.json"
EVAL_DATASET = ROOT / "datasets/sql/eval/storefront_sales_lab_eval_v1.jsonl"
BASE_URL = "http://127.0.0.1:8001"
SERVED_MODEL = "storefront-sql"

print(f"repo root: {ROOT}")
print(f"manifest:  {MANIFEST_PATH.relative_to(ROOT)}")
print(f"dataset:   {EVAL_DATASET.relative_to(ROOT)}")

repo root: /home/dahiy/repos/play-poker
manifest:  experiments/sql/qwen35_0_8b__exp048_storefront_v3_lora_r16_a32_d010.json
dataset:   datasets/sql/eval/storefront_sales_lab_eval_v1.jsonl


## 1. Load The Serving Contract

The manifest is the contract. The serving helper reads it to find the base model, adapter directory, and LoRA rank.

In [2]:
from sqlbench_lab.sql import build_vllm_serve_command, load_sql_eval_cases, load_sql_sft_manifest

manifest = load_sql_sft_manifest(MANIFEST_PATH)
adapter_dir = manifest.resolve_workspace_path(manifest.output_paths.adapter_dir)
eval_cases = load_sql_eval_cases(EVAL_DATASET)

print(f"experiment_id: {manifest.experiment_id}")
print(f"base_model:    {manifest.student.base_model}")
print(f"adapter_name:  {manifest.student.adapter_name}")
print(f"adapter_dir:   {adapter_dir.relative_to(ROOT)}")
print(f"adapter_exists:{adapter_dir.exists()}")
print(f"lora.r:        {manifest.lora.r}")
print(f"lora.alpha:    {manifest.lora.lora_alpha}")
print(f"lora.dropout:  {manifest.lora.lora_dropout}")
print(f"prompt_style:  {manifest.prompt.style}")
print(f"eval_cases:    {len(eval_cases)}")

experiment_id: qwen35_0_8b__exp048_storefront_v3_lora_r16_a32_d010
base_model:    Qwen/Qwen3.5-0.8B-Base
adapter_name:  qwen35_0_8b_storefront_v3_lora_r16_a32_d010_exp048
adapter_dir:   artifacts/sql/qwen35_0_8b__exp048_storefront_v3_lora_r16_a32_d010/adapter
adapter_exists:True
lora.r:        16
lora.alpha:    32
lora.dropout:  0.1
prompt_style:  canonical_chat
eval_cases:    12


## 2. Build The Exact vLLM Command

The repo does not hand-write this command in multiple places. `build_vllm_serve_command()` is the canonical helper. It prints a shell-safe command and also exposes the raw argv tuple.

In [3]:
serve_command = build_vllm_serve_command(
    MANIFEST_PATH,
    model_variant="adapter",
    host="127.0.0.1",
    port=8001,
    max_model_len=1536,
    gpu_memory_utilization=0.75,
    enforce_eager=True,
    flashinfer_autotune=False,
    attention_backend="TRITON_ATTN",
    served_model_name=SERVED_MODEL,
    lora_name=SERVED_MODEL,
)

env_exports = [
    "export CPATH=$HOME/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/include/python3.12",
    "export CUDA_HOME=$PWD/.venv/lib/python3.12/site-packages/nvidia/cu13",
    "export PATH=$PWD/.venv/lib/python3.12/site-packages/nvidia/cu13/bin:$PATH",
    "export VLLM_USE_FLASHINFER_SAMPLER=0",
]

print("Environment used for the local GPU run:")
print("\n".join(env_exports))
print("\nShell command:")
print("uv run --group serving " + serve_command.shell_command)
print("\nRaw argv:")
for index, part in enumerate(serve_command.command):
    print(f"{index:02d}: {part}")

Environment used for the local GPU run:
export CPATH=$HOME/.local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/include/python3.12
export CUDA_HOME=$PWD/.venv/lib/python3.12/site-packages/nvidia/cu13
export PATH=$PWD/.venv/lib/python3.12/site-packages/nvidia/cu13/bin:$PATH
export VLLM_USE_FLASHINFER_SAMPLER=0

Shell command:
uv run --group serving vllm serve Qwen/Qwen3.5-0.8B-Base --host 127.0.0.1 --port 8001 --max-model-len 1536 --gpu-memory-utilization 0.75 --enforce-eager --no-enable-flashinfer-autotune --attention-backend TRITON_ATTN --language-model-only --served-model-name storefront-sql --enable-lora --max-lora-rank 16 --lora-modules storefront-sql=/home/dahiy/repos/play-poker/artifacts/sql/qwen35_0_8b__exp048_storefront_v3_lora_r16_a32_d010/adapter

Raw argv:
00: vllm
01: serve
02: Qwen/Qwen3.5-0.8B-Base
03: --host
04: 127.0.0.1
05: --port
06: 8001
07: --max-model-len
08: 1536
09: --gpu-memory-utilization
10: 0.75
11: --enforce-eager
12: --no-enable-flashinfer-autotune
13: -

## 3. Optionally Start The Local Server

This cell is intentionally off by default because `vllm serve` is a long-running process. Set `START_VLLM_SERVER = True` when you want the notebook to launch it.

In [4]:
START_VLLM_SERVER = True
VLLM_PROCESS = None

if START_VLLM_SERVER:
    env = os.environ.copy()
    env.setdefault("CPATH", str(Path.home() / ".local/share/uv/python/cpython-3.12.12-linux-x86_64-gnu/include/python3.12"))
    env.setdefault("CUDA_HOME", str(ROOT / ".venv/lib/python3.12/site-packages/nvidia/cu13"))
    env["PATH"] = f"{ROOT / '.venv/lib/python3.12/site-packages/nvidia/cu13/bin'}:{env['PATH']}"
    env["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
    VLLM_PROCESS = subprocess.Popen(
        ["uv", "run", "--group", "serving", *serve_command.command],
        cwd=ROOT,
        env=env,
        text=True,
    )
    print(f"started vLLM pid={VLLM_PROCESS.pid}")
else:
    print("Server start skipped. Run this command in a terminal or set START_VLLM_SERVER=True:")
    print("uv run --group serving " + serve_command.shell_command)

started vLLM pid=25275


## 4. Check Endpoint Readiness

vLLM serves an OpenAI-compatible API. For this repo, the important endpoint is `/v1/completions`.

In [ ]:
REQUIRE_ENDPOINT = False


def get_json(url: str, timeout: float = 2.0) -> dict:
    with urllib.request.urlopen(url, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


def endpoint_ready(wait_seconds: float = 180.0, poll_seconds: float = 5.0) -> bool:
    deadline = time.monotonic() + wait_seconds
    last_error = None
    while True:
        try:
            models = get_json(f"{BASE_URL}/v1/models")
        except (OSError, urllib.error.URLError, TimeoutError) as exc:
            last_error = exc
            if time.monotonic() >= deadline:
                if REQUIRE_ENDPOINT:
                    raise RuntimeError(f"endpoint not ready: {BASE_URL}") from exc
                print(f"endpoint not ready yet: {BASE_URL}; last_error={type(last_error).__name__}: {last_error}")
                return False
            print(f"waiting for endpoint: {BASE_URL}")
            time.sleep(poll_seconds)
            continue
        print(json.dumps(models, indent=2)[:2000])
        return True


READY = endpoint_ready()

endpoint not ready yet: http://127.0.0.1:8001


(APIServer pid=25278) INFO 06-06 18:34:38 [vllm.py:1234] Cudagraph is disabled under eager mode
(APIServer pid=25278) INFO 06-06 18:34:38 [compilation.py:312] Enabled custom fusions: norm_quant, act_quant


(APIServer pid=25278) [transformers] `Qwen2VLImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Qwen2VLImageProcessor` instead.


(APIServer pid=25278) INFO 06-06 18:34:40 [registry.py:134] All limits of multimodal modalities supported by the model are set to 0, running in text-only mode.


## 5. Build One Real SQL Prompt

This uses the same renderer that repo eval uses. The model sees system text, schema, column notes, knowledge text, and the question. The gold SQL is printed for comparison only; it is not sent to the endpoint.

In [8]:
from sqlbench_lab.sql import build_eval_messages
from sqlbench_lab.sql.rendering import SQL_SYSTEM_PROMPT
from sqlbench_lab.sql.training import render_sql_sft_prompt

case = next(item for item in eval_cases if item.case_id == "storefront_sales_lab_eval_001")
messages = build_eval_messages(case, prompt_style=manifest.prompt.style, system_prompt=SQL_SYSTEM_PROMPT)
prompt = render_sql_sft_prompt([*messages, {"role": "assistant", "content": ""}])

print(f"case_id:    {case.case_id}")
print(f"db_path:    {case.db_path}")
print(f"question:   {case.question}")
print("\nGold SQL, not sent to model:")
print(case.gold_sql)
print(f"\nmessages:   {len(messages)}")
print(f"prompt chars: {len(prompt)}")
print("\nPrompt head:")
print(prompt[:1200])
print("\nPrompt tail:")
print(prompt[-1200:])

case_id:    storefront_sales_lab_eval_001
db_path:    datasets/sql/dbs/storefront_sales_lab/storefront_sales_lab.sqlite
question:   What discounted completed-order revenue was generated on or after 2024-04-01?

Gold SQL, not sent to model:
SELECT ROUND(SUM(T2.quantity * T2.unit_price * (1 - T1.discount_pct / 100.0)), 2) FROM orders AS T1 INNER JOIN order_items AS T2 ON T1.order_id = T2.order_id WHERE T1.status = 'completed' AND T1.order_date >= '2024-04-01'

messages:   2
prompt chars: 3009

Prompt head:
<|system|>
You are a precise text-to-SQL model. Return only the final SQL statement. Use the declared SQL dialect and stay grounded in the provided schema. Use table and column names exactly as they appear in the schema. For SQLite identifiers containing spaces, punctuation, parentheses, percent signs, hyphens, or question marks, quote the full identifier with backticks.
<|user|>
Dialect:
sqlite

Database ID:
storefront_sales_lab

Schema:
CREATE TABLE customers (
    customer_id INTEGE

## 6. Execute The Gold SQL Locally

This proves the database and expected query are real before involving the model server.

In [9]:
db_path = ROOT / case.db_path
with sqlite3.connect(db_path) as conn:
    gold_rows = conn.execute(case.gold_sql).fetchall()

print(f"db exists: {db_path.exists()} -> {db_path.relative_to(ROOT)}")
print("gold rows:")
print(gold_rows)

db exists: True -> datasets/sql/dbs/storefront_sales_lab/storefront_sales_lab.sqlite
gold rows:
[(2449.82,)]


## 7. Send One Raw Completion Request

This is the actual wire format sent to vLLM. The repo wraps this in `OpenAICompletionClient.complete()`.

In [24]:
from sqlbench_lab.sql import extract_generated_sql

payload = {
    "model": SERVED_MODEL,
    "prompt": prompt,
    "max_tokens": 128,
    "temperature": 0,
}

print("POST", f"{BASE_URL}/v1/completions")
print("Payload without full prompt:")
print(json.dumps({**payload, "prompt": f"<prompt chars={len(prompt)}>"}, indent=2))

generated_sql = None
raw_response = None
READY = READY or endpoint_ready(wait_seconds=30.0, poll_seconds=2.0)
if READY:
    request = urllib.request.Request(
        f"{BASE_URL}/v1/completions",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    started = time.perf_counter()
    with urllib.request.urlopen(request, timeout=120) as response:
        raw_response = json.loads(response.read().decode("utf-8"))
    latency = time.perf_counter() - started
    completion_text = raw_response["choices"][0]["text"]
    generated_sql = extract_generated_sql(completion_text)
    print(f"latency_seconds: {latency:.3f}")
    print("\nRaw response keys:", sorted(raw_response.keys()))
    print("\nCompletion text:")
    print(completion_text)
    print("\nExtracted SQL:")
    print(generated_sql)
else:
    curl_payload = json.dumps({**payload, "prompt": "<full rendered prompt from previous cell>"}, indent=2)
    print("Endpoint is not running. Equivalent curl shape:")
    print("curl http://127.0.0.1:8001/v1/completions -H 'Content-Type: application/json' -d '")
    print(curl_payload)
    print("'")

POST http://127.0.0.1:8001/v1/completions
Payload without full prompt:
{
  "model": "storefront-sql",
  "prompt": "<prompt chars=3009>",
  "max_tokens": 128,
  "temperature": 0
}
(APIServer pid=25278) INFO 06-06 18:49:09 [loggers.py:271] Engine 000: Avg prompt throughput: 72.1 tokens/s, Avg generation throughput: 0.5 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.6%, Prefix cache hit rate: 0.0%
(APIServer pid=25278) INFO:     127.0.0.1:34510 - "POST /v1/completions HTTP/1.1" 200 OK
latency_seconds: 10.236

Raw response keys: ['choices', 'created', 'id', 'kv_transfer_params', 'model', 'object', 'service_tier', 'system_fingerprint', 'usage']

Completion text:
SELECT ROUND(SUM(T3.quantity * T3.unit_price * (1 - T2.discount_pct / 100.0)), 2) FROM orders AS T2 INNER JOIN order_items AS T3 ON T2.order_id = T3.order_id WHERE T2.status = 'completed' AND T2.order_date >= '2024-04-01'

Extracted SQL:
SELECT ROUND(SUM(T3.quantity * T3.unit_price * (1 - T2.discount_pct / 100.0))

## 8. Score The Generated SQL

Repo eval compares execution results, not string equality. If the endpoint is up, this cell executes the generated SQL and compares it with gold.

In [25]:
from sqlbench_lab.sql import evaluate_sqlite_case

if generated_sql is None:
    print("No generated SQL because endpoint was not running.")
else:
    result = evaluate_sqlite_case(case, predicted_sql=generated_sql)
    print(f"passed: {result.passed}")
    print(f"prediction_error: {result.prediction_error}")
    print(f"gold_error:       {result.gold_error}")
    print("predicted rows:")
    print(result.predicted_rows)
    print("gold rows:")
    print(result.gold_rows)

passed: True
prediction_error: None
gold_error:       None
predicted rows:
[(2449.82,)]
gold rows:
[(2449.82,)]


## 9. Run The Repo Endpoint Gates

The single request above proves the path. These commands are the real gates used for endpoint quality and local stress.

In [26]:
eval_command = [
    "uv", "run", "python", "-m", "sqlbench_lab.cli", "sql", "eval",
    "--manifest", str(MANIFEST_PATH.relative_to(ROOT)),
    "--model", "adapter",
    "--dataset", str(EVAL_DATASET.relative_to(ROOT)),
    "--openai-base-url", BASE_URL,
    "--openai-model", SERVED_MODEL,
    "--result-label", "vllm_eval",
    "--max-new-tokens", "128",
]
load_command = [
    "uv", "run", "python", "-m", "sqlbench_lab.cli", "sql", "openai-load-test",
    "--manifest", str(MANIFEST_PATH.relative_to(ROOT)),
    "--model", "adapter",
    "--dataset", str(EVAL_DATASET.relative_to(ROOT)),
    "--openai-base-url", BASE_URL,
    "--openai-model", SERVED_MODEL,
    "--requests", "32",
    "--concurrency", "8",
    "--output", "artifacts/sql/qwen35_0_8b__exp048_storefront_v3_lora_r16_a32_d010/vllm_load_c8.json",
    "--max-new-tokens", "128",
]

print("Endpoint eval command:")
print(shlex.join(eval_command))
print("\nLoad probe command:")
print(shlex.join(load_command))

Endpoint eval command:
uv run python -m sqlbench_lab.cli sql eval --manifest experiments/sql/qwen35_0_8b__exp048_storefront_v3_lora_r16_a32_d010.json --model adapter --dataset datasets/sql/eval/storefront_sales_lab_eval_v1.jsonl --openai-base-url http://127.0.0.1:8001 --openai-model storefront-sql --result-label vllm_eval --max-new-tokens 128

Load probe command:
uv run python -m sqlbench_lab.cli sql openai-load-test --manifest experiments/sql/qwen35_0_8b__exp048_storefront_v3_lora_r16_a32_d010.json --model adapter --dataset datasets/sql/eval/storefront_sales_lab_eval_v1.jsonl --openai-base-url http://127.0.0.1:8001 --openai-model storefront-sql --requests 32 --concurrency 8 --output artifacts/sql/qwen35_0_8b__exp048_storefront_v3_lora_r16_a32_d010/vllm_load_c8.json --max-new-tokens 128


## 10. Where KV Cache Lives

KV cache is not a repo artifact and not a normal file. vLLM allocates it as GPU tensors during server startup. These snippets are from the installed local vLLM package in `.venv/`.

In [27]:
SNIPPETS = [
    (ROOT / ".venv/lib/python3.12/site-packages/vllm/v1/kv_cache_interface.py", 813, 847),
    (ROOT / ".venv/lib/python3.12/site-packages/vllm/v1/worker/gpu_model_runner.py", 6850, 6880),
    (ROOT / ".venv/lib/python3.12/site-packages/vllm/v1/worker/gpu_model_runner.py", 7052, 7105),
    (ROOT / ".venv/lib/python3.12/site-packages/vllm/v1/worker/utils.py", 460, 516),
    (ROOT / ".venv/lib/python3.12/site-packages/vllm/v1/core/kv_cache_manager.py", 236, 409),
    (ROOT / ".venv/lib/python3.12/site-packages/vllm/v1/core/single_type_kv_cache_manager.py", 71, 74),
    (ROOT / ".venv/lib/python3.12/site-packages/vllm/v1/core/single_type_kv_cache_manager.py", 243, 270),
    (ROOT / ".venv/lib/python3.12/site-packages/vllm/v1/core/single_type_kv_cache_manager.py", 338, 352),
]


def print_file_snippet(path: Path, start: int, end: int) -> None:
    print("=" * 100)
    print(path.relative_to(ROOT), f"lines {start}-{end}")
    if not path.exists():
        print("missing local file; install vLLM with `uv sync --group serving`")
        return
    lines = path.read_text(encoding="utf-8").splitlines()
    for line_number in range(start, min(end, len(lines)) + 1):
        print(f"{line_number:5d}  {lines[line_number - 1]}")


for snippet in SNIPPETS:
    print_file_snippet(*snippet)

.venv/lib/python3.12/site-packages/vllm/v1/kv_cache_interface.py lines 813-847
  813  class KVCacheTensor:
  814      """
  815      A class for specifying how the workers should initialize the KV cache.
  816      """
  817  
  818      size: int  # size of the KV cache tensor in bytes
  819      shared_by: list[str]  # layer names that share the same KV cache tensor
  820  
  821  
  822  @dataclass
  823  class KVCacheGroupSpec:
  824      """
  825      Represents a group of model layers that share the same KV cache block table.
  826      These layers are regarded as one layer in the KV cache manager.
  827      """
  828  
  829      # The names of model layers in this group
  830      layer_names: list[str]
  831      # The KV cache spec of this manager layer
  832      kv_cache_spec: KVCacheSpec
  833      # Whether this group contains EAGLE/MTP draft attention layers.
  834      is_eagle_group: bool = False
  835  
  836  
  837  @dataclass
  838  class KVCacheConfig:
  839   

(APIServer pid=25278) INFO 06-06 18:49:19 [loggers.py:271] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 7.5 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%
(APIServer pid=25278) INFO 06-06 18:49:26 [loggers.py:271] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 0.0%
(APIServer pid=25278) INFO 06-06 22:32:13 [loggers.py:271] Engine 000: Avg prompt throughput: 72.1 tokens/s, Avg generation throughput: 0.6 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.6%, Prefix cache hit rate: 0.0%
(APIServer pid=25278) INFO 06-06 22:32:23 [loggers.py:271] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 7.3 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.6%, Prefix cache hit rate: 0.0%
(APIServer pid=25278) INFO:     127.0.0.1:33124 - "

(APIServer pid=25278) INFO:     Shutting down
(APIServer pid=25278) INFO:     Waiting for application shutdown.


(EngineCore pid=25458) INFO 06-06 22:36:30 [core.py:1266] Shutdown initiated (timeout=0)
(EngineCore pid=25458) INFO 06-06 22:36:30 [core.py:1289] Shutdown complete
(APIServer pid=25278) INFO 06-06 22:36:30 [launcher.py:137] Shutting down FastAPI HTTP server.


(APIServer pid=25278) INFO:     Shutting down
(APIServer pid=25278) INFO:     Waiting for application shutdown.
(APIServer pid=25278) INFO:     Application shutdown complete.
/usr/lib/python3.12/multiprocessing/resource_tracker.py:254: UserWarning: resource_tracker: There appear to be 1 leaked semaphore objects to clean up at shutdown
  warnings.warn('resource_tracker: There appear to be %d '


## 11. KV Cache In Cave-Man Terms

The server starts. vLLM loads model weights into GPU memory. Then it reserves another big area of GPU memory for KV cache blocks.

A request comes in. vLLM gives that request some block IDs. While the model reads the prompt, every attention layer writes key/value memory into those blocks. When the model generates the next token, it reads the saved blocks instead of recomputing the full prompt.

More requests come in. PagedAttention means each request can get different block IDs instead of one giant continuous chunk. When a request finishes, vLLM frees its block IDs so another request can use them.

For the local Exp048 run, vLLM reported 230,912 KV-cache tokens and about 150x theoretical concurrency at 1,536 tokens/request. The stress test admitted c160, but useful interactive latency was closer to c8-c16.